# Preparação e Transformação dos Dados

**Projeto:** ANEEL - Energia em Risco: Análise de Dados de Continuidade Elétrica e Previsão de Risco Regulatório 

**Metodologia:** CRISP-DM

**Período de Análise:** 2021 – 2025

---

## Visão Geral

Este notebook consolida a **Fase 3 do CRISP-DM - Preparação dos Dados (Data Preparation)** com a definição do alvo, seleção de atributos, tratamento de nulos e outliers, codificação de categorias, agregações e controle de vazamento de dados (data leakage).

---

## Sumário

1. Configurações do ambiente  
2. Objetivo e definição do problema  
3. Seleção das fontes e granularidade  
4. Construção do alvo temporal  
5. Engenharia de atributos históricos  
6. Tratamento da qualidade dos dados  
7. Codificação e transformação das variáveis  
8. Divisão temporal e prevenção de vazamento  
9. Desbalanceamento da variável alvo  
10. Base final para modelagem  
11. Limitações e decisões para a modelagem  
12. Síntese da preparação dos dados  

---

## 1. Configurações do ambiente

In [45]:
# Importações
import os
import sys
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from IPython.display import display

# Configurações de diretórios
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PROCESSED_DIR, ATR_PATH
from src.data.constants import ANO_INICIO, ANO_FIM

MODEL_DATA_DIR = PROCESSED_DIR / "modeling"
MODEL_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Configurações de visualização
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", lambda value: f"{value:.2f}")
sns.set_theme(style="whitegrid")

# Conexão DuckDB
con = duckdb.connect()

# Tabelas Gold utilizadas na análise exploratória
FACT_CONT_PATH = PROCESSED_DIR / "fato_continuidade.parquet"
FACT_CAUSA_PATH = PROCESSED_DIR / "fato_causa_mensal.parquet"
DIM_DATA_PATH = PROCESSED_DIR / "dim_data.parquet"
DIM_DISTRIBUIDORA_PATH = PROCESSED_DIR / "dim_distribuidora.parquet"
DIM_CONJUNTO_PATH = PROCESSED_DIR / "dim_conjunto.parquet"
DIM_INDICADOR_PATH = PROCESSED_DIR / "dim_indicador.parquet"
DIM_TIPO_PATH = PROCESSED_DIR / "dim_tipo_interrupcao.parquet"
DIM_MOTIVO_PATH = PROCESSED_DIR / "dim_motivo_interrupcao.parquet"
DIM_CAUSA_PATH = PROCESSED_DIR / "dim_causa_interrupcao.parquet"

required_paths = [
    FACT_CONT_PATH, FACT_CAUSA_PATH, DIM_DATA_PATH, DIM_DISTRIBUIDORA_PATH,
    DIM_CONJUNTO_PATH, DIM_INDICADOR_PATH, DIM_TIPO_PATH, DIM_MOTIVO_PATH,
    DIM_CAUSA_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Arquivos processed ausentes:\n" + "\n".join(missing_paths))

print(f"Período analisado: {ANO_INICIO}-{ANO_FIM}")

Período analisado: 2021-2025


---

## 2. Objetivo e definição do problema

O objetivo desta etapa é preparar uma base supervisionada para prever o risco de transgressão regulatória no período seguinte, utilizando informações disponíveis até o período atual.

**Unidade de análise:** conjunto elétrico, indicador (`DEC` ou `FEC`) e período.

**Variável alvo:** `UltrapassouLimite` deslocada para o próximo período. Assim, as características calculadas até o período `t` serão utilizadas para prever a transgressão em `t + 1`.

**Regra principal:** nenhuma variável do período futuro pode ser utilizada na construção das características do período atual, evitando vazamento temporal e garantindo que a base reproduza o cenário real de previsão.

---

## 3. Seleção das fontes e granularidade


A base de modelagem será construída a partir das tabelas Gold, mantendo a granularidade principal de conjunto, indicador e período.

| Tabela | Descrição |
|---|---|
|`fato_continuidade` | valores DEC/FEC, limites regulatórios e transgressões; fonte do alvo e das variáveis históricas |
| `fato_causa_mensal` | quantidade, duração e impacto das interrupções; fonte dos atributos operacionais agregados |
| `dim_data` | calendário e ordenação temporal |
| `dim_conjunto` | identificação do conjunto e região |
| `dim_distribuidora` | identificação da distribuidora |
| `dim_indicador` | identificação de DEC e FEC |
| `atributos.parquet` | quantidade cadastral estimada de consumidores pelos indicadores `NUCT*`, quando houver cobertura compatível |

As dimensões serão utilizadas para enriquecer os fatos, sem alterar a unidade de observação do conjunto, indicador e período.

In [46]:
# Inventário das fontes selecionadas para a base de modelagem
source_specs = {
    "fato_continuidade": (FACT_CONT_PATH, "Fato principal: DEC/FEC, limites e transgressões"),
    "fato_causa_mensal": (FACT_CAUSA_PATH, "Fato operacional: interrupções, duração e impacto"),
    "dim_data": (DIM_DATA_PATH, "Dimensão temporal"),
    "dim_conjunto": (DIM_CONJUNTO_PATH, "Dimensão cadastral e regional"),
    "dim_distribuidora": (DIM_DISTRIBUIDORA_PATH, "Dimensão da distribuidora"),
    "dim_indicador": (DIM_INDICADOR_PATH, "Dimensão dos indicadores DEC/FEC"),
}

source_rows = []
for source_name, (source_path, purpose) in source_specs.items():
    source_schema = con.execute(
        "DESCRIBE SELECT * FROM read_parquet(?)",
        [str(source_path)],
    ).fetchall()
    source_rows.append({
        "fonte": source_name,
        "linhas": int(con.execute(
            "SELECT COUNT(*) FROM read_parquet(?)",
            [str(source_path)],
        ).fetchone()[0]),
        "colunas": len(source_schema),
        "finalidade": purpose,
    })

df_source_inventory = pd.DataFrame(source_rows)
display(df_source_inventory)

,fonte,linhas,colunas,finalidade
0,fato_continuidade,375082,10,"Fato principal: DEC/FEC, limites e transgressões"
1,fato_causa_mensal,3002279,16,"Fato operacional: interrupções, duração e impacto"
2,dim_data,1826,7,Dimensão temporal
3,dim_conjunto,3824,9,Dimensão cadastral e regional
4,dim_distribuidora,105,3,Dimensão da distribuidora
5,dim_indicador,2,4,Dimensão dos indicadores DEC/FEC


---

## 4. Construção do alvo temporal


O **alvo** será formado pela transgressão do período seguinte. Para cada conjunto e indicador, os registros são ordenados cronologicamente e a variável `UltrapassouLimite` é deslocada com `LEAD`.

Somente observações cujo próximo registro corresponda ao mês imediatamente seguinte serão mantidas. Registros sem período seguinte ou com lacunas temporais não entram na base final desta etapa.

In [47]:
# Construção do alvo de transgressão no mês seguinte
query_target = f"""
WITH ordered AS (
    SELECT
        f.ConjuntoKey,
        f.IndicadorKey,
        i.SigIndicador,
        d.Data,
        f.VlrIndicador,
        f.VlrLimite,
        f.UltrapassouLimite AS transgressao_atual,
        LEAD(d.Data) OVER (
            PARTITION BY f.ConjuntoKey, f.IndicadorKey
            ORDER BY d.Data
        ) AS data_seguinte,
        LEAD(f.UltrapassouLimite) OVER (
            PARTITION BY f.ConjuntoKey, f.IndicadorKey
            ORDER BY d.Data
        ) AS ultrapassou_limite_seguinte
    FROM read_parquet('{FACT_CONT_PATH}') f
    JOIN read_parquet('{DIM_DATA_PATH}') d
      ON f.DataKey = d.DataKey
    JOIN read_parquet('{DIM_INDICADOR_PATH}') i
      ON f.IndicadorKey = i.IndicadorKey
)
SELECT
    ConjuntoKey,
    IndicadorKey,
    SigIndicador,
    Data,
    VlrIndicador,
    VlrLimite,
    transgressao_atual,
    data_seguinte,
    CAST(ultrapassou_limite_seguinte AS INTEGER) AS alvo_transgressao_seguinte
FROM ordered
WHERE data_seguinte IS NOT NULL
  AND date_diff('month', Data, data_seguinte) = 1
ORDER BY ConjuntoKey, IndicadorKey, Data
"""

df_target = con.execute(query_target).df()
display(df_target.head(10))
print(f"Observações com alvo temporal válido: {len(df_target):,}")

,ConjuntoKey,IndicadorKey,SigIndicador,Data,VlrIndicador,VlrLimite,transgressao_atual,data_seguinte,alvo_transgressao_seguinte
0,1,1,DEC,2021-01-01,2.05,35.00,0,2021-02-01,0
1,1,1,DEC,2021-02-01,0.39,35.00,0,2021-03-01,0
2,1,1,DEC,2021-03-01,2.53,35.00,0,2021-04-01,0
3,1,1,DEC,2021-04-01,4.47,35.00,0,2021-05-01,0
4,1,1,DEC,2021-05-01,0.27,35.00,0,2021-06-01,0
5,1,1,DEC,2021-06-01,0.58,35.00,0,2021-07-01,0
6,1,1,DEC,2021-07-01,0.31,35.00,0,2021-08-01,0
7,1,1,DEC,2021-08-01,0.23,35.00,0,2021-09-01,0
8,1,1,DEC,2021-09-01,0.30,35.00,0,2021-10-01,0
9,1,1,DEC,2021-10-01,0.07,35.00,0,2021-11-01,0


Observações com alvo temporal válido: 367,282


In [48]:
# Validação de unicidade no df_target
duplicatas = df_target.duplicated(subset=["ConjuntoKey", "IndicadorKey", "Data"]).sum()
print(f"Duplicatas encontradas: {duplicatas}")
assert duplicatas == 0, "Atenção: existem duplicatas de data por conjunto e indicador!"

Duplicatas encontradas: 0


---

## 5. Engenharia de atributos históricos

Os atributos serão calculados dentro de cada conjunto e indicador, respeitando a ordem temporal. As janelas incluem o período atual e períodos anteriores, pois essas informações já estariam disponíveis no momento da previsão do mês seguinte.

Serão considerados valores atuais, defasagens, médias móveis, histórico de transgressões, sazonalidade, região e distribuidora. O alvo futuro não participa de nenhuma janela de atributos.

In [49]:
# Engenharia de atributos históricos sem utilizar o alvo futuro
query_features = f"""
WITH base AS (
    SELECT
        f.ConjuntoKey,
        f.IndicadorKey,
        i.SigIndicador,
        d.Data,
        d.Ano,
        d.MesNumero,
        c.Regiao,
        dist.SigAgente,
        f.VlrIndicador,
        f.VlrLimite,
        f.PercentualDoLimite,
        CAST(f.UltrapassouLimite AS INTEGER) AS transgressao_atual,
        LEAD(d.Data) OVER (
            PARTITION BY f.ConjuntoKey, f.IndicadorKey
            ORDER BY d.Data
        ) AS data_seguinte,
        LEAD(CAST(f.UltrapassouLimite AS INTEGER)) OVER (
            PARTITION BY f.ConjuntoKey, f.IndicadorKey
            ORDER BY d.Data
        ) AS alvo_transgressao_seguinte,
        LAG(f.VlrIndicador, 1) OVER (
            PARTITION BY f.ConjuntoKey, f.IndicadorKey
            ORDER BY d.Data
        ) AS vlr_indicador_lag_1,
        LAG(f.VlrIndicador, 3) OVER (
            PARTITION BY f.ConjuntoKey, f.IndicadorKey
            ORDER BY d.Data
        ) AS vlr_indicador_lag_3,
        AVG(f.VlrIndicador) OVER (
            PARTITION BY f.ConjuntoKey, f.IndicadorKey
            ORDER BY d.Data
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS vlr_indicador_media_3,
        SUM(CAST(f.UltrapassouLimite AS INTEGER)) OVER (
            PARTITION BY f.ConjuntoKey, f.IndicadorKey
            ORDER BY d.Data
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS transgressoes_ultimos_3
    FROM read_parquet('{FACT_CONT_PATH}') f
    JOIN read_parquet('{DIM_DATA_PATH}') d
      ON f.DataKey = d.DataKey
    JOIN read_parquet('{DIM_INDICADOR_PATH}') i
      ON f.IndicadorKey = i.IndicadorKey
    JOIN read_parquet('{DIM_CONJUNTO_PATH}') c
      ON f.ConjuntoKey = c.ConjuntoKey
    JOIN read_parquet('{DIM_DISTRIBUIDORA_PATH}') dist
      ON f.DistribuidoraKey = dist.DistribuidoraKey
)
SELECT
    *,
    date_diff('month', Data, data_seguinte) AS distancia_meses
FROM base
WHERE data_seguinte IS NOT NULL
  AND date_diff('month', Data, data_seguinte) = 1
ORDER BY ConjuntoKey, IndicadorKey, Data
"""

df_features = con.execute(query_features).df()
display(df_features.head())
print(f"Observações com atributos e alvo válidos: {len(df_features):,}")

,ConjuntoKey,IndicadorKey,SigIndicador,Data,Ano,MesNumero,Regiao,SigAgente,VlrIndicador,VlrLimite,PercentualDoLimite,transgressao_atual,data_seguinte,alvo_transgressao_seguinte,vlr_indicador_lag_1,vlr_indicador_lag_3,vlr_indicador_media_3,transgressoes_ultimos_3,distancia_meses
0,1,1,DEC,2021-01-01,2021,1,SUL,CERGAPA,2.05,35.00,0.06,0,2021-02-01,0,NaN,NaN,2.05,0.00,1
1,1,1,DEC,2021-02-01,2021,2,SUL,CERGAPA,0.39,35.00,0.01,0,2021-03-01,0,2.05,NaN,1.22,0.00,1
2,1,1,DEC,2021-03-01,2021,3,SUL,CERGAPA,2.53,35.00,0.07,0,2021-04-01,0,0.39,NaN,1.66,0.00,1
3,1,1,DEC,2021-04-01,2021,4,SUL,CERGAPA,4.47,35.00,0.13,0,2021-05-01,0,2.53,2.05,2.46,0.00,1
4,1,1,DEC,2021-05-01,2021,5,SUL,CERGAPA,0.27,35.00,0.01,0,2021-06-01,0,4.47,0.39,2.42,0.00,1


Observações com atributos e alvo válidos: 367,282


---

## 6. Tratamento da qualidade dos dados

A qualidade será avaliada antes das transformações finais. As primeiras defasagens podem ser nulas porque não existe histórico anterior suficiente, portanto, esses valores não representam erro de origem.

A estratégia adotada é:

- remover somente registros sem alvo temporal válido, já filtrados na etapa anterior;
- verificar duplicidades na chave conjunto, indicador e data;
- preservar os nulos das defasagens durante a preparação exploratória;
- calcular eventuais medianas e imputações somente com dados do treino, evitando vazamento entre períodos;
- investigar valores extremos antes de aplicar qualquer transformação.

In [50]:
# Diagnóstico de nulos, duplicidades e distribuição do alvo
key_columns = ["ConjuntoKey", "IndicadorKey", "Data"]
numeric_feature_columns = df_features.select_dtypes(include="number").columns

quality_rows = []
df_feature_quality = pd.DataFrame(
    {
        "coluna": df_features.columns,
        "tipo": df_features.dtypes.astype(str).values,
        "nulos": df_features.isna().sum().values,
        "percentual_nulos": (df_features.isna().mean() * 100).round(2).values,
    }
)

duplicate_count = int(df_features.duplicated(key_columns).sum())
target_distribution = (
    df_features["alvo_transgressao_seguinte"]
    .value_counts(dropna=False)
    .rename_axis("alvo_transgressao_seguinte")
    .reset_index(name="observacoes")
)

display(df_feature_quality)
display(target_distribution)
print(f"Duplicidades na chave {key_columns}: {duplicate_count}")
print(f"Colunas numéricas avaliadas: {len(numeric_feature_columns)}")

,coluna,tipo,nulos,percentual_nulos
0,ConjuntoKey,int64,0,0.00
1,IndicadorKey,int8,0,0.00
2,SigIndicador,str,0,0.00
3,Data,datetime64[us],0,0.00
4,Ano,int16,0,0.00
5,MesNumero,int8,0,0.00
6,Regiao,str,0,0.00
7,SigAgente,str,0,0.00
8,VlrIndicador,float64,0,0.00
9,VlrLimite,float64,750,0.20


,alvo_transgressao_seguinte,observacoes
0,0,367192
1,1,90


Duplicidades na chave ['ConjuntoKey', 'IndicadorKey', 'Data']: 0
Colunas numéricas avaliadas: 14


In [51]:
# Alvo por indicado DEC/FEC

target_by_indicator = (
    df_features.groupby(["SigIndicador", "alvo_transgressao_seguinte"])
    .size()
    .unstack(fill_value=0)
)
target_by_indicator["taxa_positiva_%"] = round(
    target_by_indicator[1] / (target_by_indicator[0] + target_by_indicator[1]) * 100, 3
)
display(target_by_indicator)

alvo_transgressao_seguinte,0,1,taxa_positiva_%
SigIndicador,,,
DEC,183691,78,0.04
FEC,183501,12,0.01


---

## 7. Codificação e transformação das variáveis

As variáveis serão separadas entre numéricas e categóricas. O mês será representado também por componentes cíclicos, preservando a proximidade entre dezembro e janeiro.

As colunas `data_seguinte` e `distancia_meses` são auxiliares da construção do alvo e não serão utilizadas como atributos. A codificação das categorias e a imputação dos nulos serão ajustadas somente no conjunto de treino.

---

In [52]:
# 1. Agregação cadastral de unidades consumidoras (NUCT*)
query_attributes = f"""
WITH consumidores AS (
    SELECT
        IdeConjunto,
        SUM(TRY_CAST(REPLACE(VlrIndiceEnviado, ',', '.') AS DOUBLE)) AS unidades_consumidoras
    FROM read_parquet('{ATR_PATH}')
    WHERE SigIndicador LIKE 'NUCT%'
    GROUP BY IdeConjunto
)
SELECT
    c.ConjuntoKey,
    COALESCE(cons.unidades_consumidoras, 0.0) AS unidades_consumidoras,
    CASE
        WHEN COALESCE(cons.unidades_consumidoras, 0.0) < 1000 THEN 'Até 1.000'
        WHEN COALESCE(cons.unidades_consumidoras, 0.0) < 10000 THEN '1.000 a 10.000'
        WHEN COALESCE(cons.unidades_consumidoras, 0.0) < 50000 THEN '10.000 a 50.000'
        ELSE '50.000 ou mais'
    END AS faixa_consumidores
FROM read_parquet('{DIM_CONJUNTO_PATH}') c
LEFT JOIN consumidores cons
  ON c.IdeConjunto = cons.IdeConjunto
"""

df_attributes = con.execute(query_attributes).df()

# 2. Descarte de colunas de controle temporal e junção cadastral
model_drop_columns = ["data_seguinte", "distancia_meses"]
df_model_base = df_features.drop(columns=model_drop_columns).copy()

df_model_base = df_model_base.merge(df_attributes, on="ConjuntoKey", how="left")
df_model_base["unidades_consumidoras"] = df_model_base["unidades_consumidoras"].fillna(
    0.0
)
df_model_base["faixa_consumidores"] = df_model_base["faixa_consumidores"].fillna(
    "Até 1.000"
)

# 3. Transformações sazonais cíclicas e alvo
df_model_base["mes_seno"] = np.sin(2 * np.pi * df_model_base["MesNumero"] / 12)
df_model_base["mes_cosseno"] = np.cos(2 * np.pi * df_model_base["MesNumero"] / 12)
df_model_base["alvo"] = df_model_base.pop("alvo_transgressao_seguinte").astype("int8")

# 4. Definição das colunas categóricas para o One-Hot Encoding
categorical_columns = ["SigIndicador", "Regiao", "SigAgente", "faixa_consumidores"]

non_numeric_features = categorical_columns + [
    "Data",
    "alvo",
    "ConjuntoKey",
    "IndicadorKey",
    "MesNumero",
]
numeric_columns = [
    column_name
    for column_name in df_model_base.columns
    if column_name not in non_numeric_features
]

print(f"Colunas categóricas: {categorical_columns}")
print(f"Colunas numéricas: {len(numeric_columns)}")
print(f"Dimensão da base: {df_model_base.shape}")

Colunas categóricas: ['SigIndicador', 'Regiao', 'SigAgente', 'faixa_consumidores']
Colunas numéricas: 12
Dimensão da base: (367282, 21)


In [53]:
df_model_base.info()

<class 'pandas.DataFrame'>
RangeIndex: 367282 entries, 0 to 367281
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   ConjuntoKey              367282 non-null  int64         
 1   IndicadorKey             367282 non-null  int8          
 2   SigIndicador             367282 non-null  str           
 3   Data                     367282 non-null  datetime64[us]
 4   Ano                      367282 non-null  int16         
 5   MesNumero                367282 non-null  int8          
 6   Regiao                   367282 non-null  str           
 7   SigAgente                367282 non-null  str           
 8   VlrIndicador             367282 non-null  float64       
 9   VlrLimite                366532 non-null  float64       
 10  PercentualDoLimite       366532 non-null  float64       
 11  transgressao_atual       367282 non-null  int32         
 12  vlr_indicador_lag_1      35

In [54]:
display(df_model_base.head(5).T)

,0,1,2,3,4
ConjuntoKey,1,1,1,1,1
IndicadorKey,1,1,1,1,1
SigIndicador,DEC,DEC,DEC,DEC,DEC
Data,2021-01-01 00:00:00,2021-02-01 00:00:00,2021-03-01 00:00:00,2021-04-01 00:00:00,2021-05-01 00:00:00
Ano,2021,2021,2021,2021,2021
MesNumero,1,2,3,4,5
Regiao,SUL,SUL,SUL,SUL,SUL
SigAgente,CERGAPA,CERGAPA,CERGAPA,CERGAPA,CERGAPA
VlrIndicador,2.05,0.39,2.53,4.47,0.27
VlrLimite,35.00,35.00,35.00,35.00,35.00


## 8. Divisão temporal e prevenção de vazamento


A separação será temporal, sem embaralhar as observações. O período de treino será usado para ajustar imputações, codificação e demais transformações; validação e teste serão apenas avaliados com esses parâmetros.

In [55]:
# Separação temporal da base de modelagem
train_end_year = ANO_FIM - 2
validation_year = ANO_FIM - 1
test_year = ANO_FIM

train_mask = df_model_base["Ano"] <= train_end_year
validation_mask = df_model_base["Ano"] == validation_year
test_mask = df_model_base["Ano"] == test_year

train_data = df_model_base.loc[train_mask].copy()
validation_data = df_model_base.loc[validation_mask].copy()
test_data = df_model_base.loc[test_mask].copy()

total_observations = len(df_model_base)
split_summary = pd.DataFrame(
    [
        {
            "particao": "treino",
            "periodo": f"{ANO_INICIO}-{train_end_year}",
            "observacoes": len(train_data),
            "percentual_base": round(100 * len(train_data) / total_observations, 2),
            "transgressoes": int(train_data["alvo"].sum()),
        },
        {
            "particao": "validacao",
            "periodo": str(validation_year),
            "observacoes": len(validation_data),
            "percentual_base": round(
                100 * len(validation_data) / total_observations, 2
            ),
            "transgressoes": int(validation_data["alvo"].sum()),
        },
        {
            "particao": "teste",
            "periodo": str(test_year),
            "observacoes": len(test_data),
            "percentual_base": round(100 * len(test_data) / total_observations, 2),
            "transgressoes": int(test_data["alvo"].sum()),
        },
    ]
)

display(split_summary)
assert (len(train_data) + len(validation_data) + len(test_data)) == total_observations
assert np.isclose(split_summary["percentual_base"].sum(), 100.0, atol=0.05)
print("Divisão temporal validada sem sobreposição entre partições.")

,particao,periodo,observacoes,percentual_base,transgressoes
0,treino,2021-2023,223184,60.77,70
1,validacao,2024,75072,20.44,14
2,teste,2025,69026,18.79,6


Divisão temporal validada sem sobreposição entre partições.


Para o período disponível, foi adotada a seguinte divisão:

- **Treino:** 2021–2023 (~61%);
- **Validação:** 2024 (~20%);
- **Teste:** 2025 (~19%).

Essa estratégia simula o uso do modelo em produção: aprender com o passado e avaliar em períodos posteriores.

---

## 9. Desbalanceamento da variável alvo

As transgressões são eventos raros na base. O treinamento deverá considerar pesos de classe ou outra estratégia adequada, sem aplicar oversampling antes da separação temporal.

A avaliação deve priorizar `recall`, `precision`, `F1`, `matriz de confusão` e `PR-AUC`. A acurácia isolada pode ser enganosa quando a maioria das observações pertence à classe sem transgressão.

In [56]:
# Distribuição do alvo e peso de classe calculado no treino
imbalance_rows = []
for partition_name, partition_data in {
    "treino": train_data,
    "validacao": validation_data,
    "teste": test_data,
}.items():
    positives = int(partition_data["alvo"].sum())
    observations = len(partition_data)
    imbalance_rows.append(
        {
            "particao": partition_name,
            "observacoes": observations,
            "sem_transgressao": observations - positives,
            "com_transgressao": positives,
            "percentual_transgressao": round(100 * positives / observations, 4),
        }
    )

imbalance_summary = pd.DataFrame(imbalance_rows)
train_positives = int(train_data["alvo"].sum())
train_negatives = len(train_data) - train_positives
scale_pos_weight = train_negatives / train_positives if train_positives else 1.0
class_weight = {
    0: 1.0,
    1: scale_pos_weight,
}

display(imbalance_summary)
print(f"Pesos sugeridos para o treino: {{0: 1.0, 1: {class_weight[1]:.2f}}}")

,particao,observacoes,sem_transgressao,com_transgressao,percentual_transgressao
0,treino,223184,223114,70,0.03
1,validacao,75072,75058,14,0.02
2,teste,69026,69020,6,0.01


Pesos sugeridos para o treino: {0: 1.0, 1: 3187.34}


---

## 10. Base final para modelagem


A base final não utilizará identificadores técnicos nem a data bruta como atributos. As variáveis categóricas serão codificadas a partir das categorias observadas no treino, e os nulos numéricos serão imputados pela mediana calculada no treino.

As mesmas colunas e os mesmos parâmetros serão aplicados à validação e ao teste, garantindo comparabilidade e evitando vazamento de informação.

In [57]:
# Preparação final das matrizes de modelagem
identifier_columns = ["ConjuntoKey", "IndicadorKey", "Data", "MesNumero", "alvo"]

X_train_raw = train_data.drop(columns=identifier_columns)
X_validation_raw = validation_data.drop(columns=identifier_columns)
X_test_raw = test_data.drop(columns=identifier_columns)
y_train = train_data["alvo"].copy()
y_validation = validation_data["alvo"].copy()
y_test = test_data["alvo"].copy()

X_train = pd.get_dummies(
    X_train_raw,
    columns=categorical_columns,
    dtype="int8",
)
X_validation = pd.get_dummies(
    X_validation_raw,
    columns=categorical_columns,
    dtype="int8",
).reindex(columns=X_train.columns, fill_value=0)
X_test = pd.get_dummies(
    X_test_raw,
    columns=categorical_columns,
    dtype="int8",
).reindex(columns=X_train.columns, fill_value=0)

numeric_model_columns = X_train.select_dtypes(include="number").columns
train_medians = X_train[numeric_model_columns].median()
for matrix in [X_train, X_validation, X_test]:
    matrix[numeric_model_columns] = matrix[numeric_model_columns].fillna(train_medians)

final_matrix_summary = pd.DataFrame(
    [
        {
            "particao": "treino",
            "observacoes": len(X_train),
            "atributos": X_train.shape[1],
        },
        {
            "particao": "validacao",
            "observacoes": len(X_validation),
            "atributos": X_validation.shape[1],
        },
        {"particao": "teste", "observacoes": len(X_test), "atributos": X_test.shape[1]},
    ]
)

display(final_matrix_summary)
print(f"Nulos em X_train: {int(X_train.isna().sum().sum())}")
print(f"Nulos em X_validation: {int(X_validation.isna().sum().sum())}")
print(f"Nulos em X_test: {int(X_test.isna().sum().sum())}")
assert list(X_train.columns) == list(X_validation.columns) == list(X_test.columns)

,particao,observacoes,atributos
0,treino,223184,129
1,validacao,75072,129
2,teste,69026,129


Nulos em X_train: 0
Nulos em X_validation: 0
Nulos em X_test: 0


In [58]:
# Dicionários de matrizes (alvo e features)
features_dict = {
    "X_train": X_train,
    "X_validation": X_validation,
    "X_test": X_test,
}

targets_dict = {
    "y_train": y_train,
    "y_validation": y_validation,
    "y_test": y_test,
}

relative_model_dir = os.path.relpath(MODEL_DATA_DIR)

# Persistência das matrizes preparadas para o notebook de modelagem

for matrix_name, matrix in {
    "X_train": X_train,
    "X_validation": X_validation,
    "X_test": X_test,
}.items():
    matrix.to_parquet(MODEL_DATA_DIR / f"{matrix_name}.parquet", index=False)

for target_name, target in {
    "y_train": y_train,
    "y_validation": y_validation,
    "y_test": y_test,
}.items():
    target.to_frame(name="alvo").to_parquet(
        MODEL_DATA_DIR / f"{target_name}.parquet",
        index=False,
    )

print(f"\nTodas as matrizes foram persistidas com sucesso em '{relative_model_dir}'.")


Todas as matrizes foram persistidas com sucesso em '..\data\processed\modeling'.


---

## 11. Limitações e decisões para a modelagem

- A variável alvo é extremamente rara: A ocorrência de transgressões regulatórias é um evento raro frente ao volume consolidado. A performance dos classificadores será orientada estritamente por métricas sensíveis à classe minoritária (`PR-AUC`, `Precision-Recall Curve`, `Recall`, `F1-Score`), descartando a acurácia global como critério de seleção.

- A divisão estrita por janelas anuais preserva a cronologia e elimina vazamento de dados (data leakage), mas restringe o volume absoluto de eventos positivos nos conjuntos de validação e teste.

- As primeiras observações de cada conjunto contêm nulos estruturais decorrentes do tamanho das janelas móveis (lags de 1 e 3 períodos). A imputação por mediana foi ajustada estritamente no particionamento de treino para impedir contaminação temporal.

- Os indicadores `NUCT*` têm cobertura parcial e representam uma estimativa cadastral de consumidores, não uma medida direta de densidade espacial.

- Valores extremos de DEC/FEC e interrupções devem ser contextualizados, não removidos automaticamente.

---

## 12. Síntese da preparação dos dados


A base foi preparada para prever a transgressão regulatória do período seguinte, mantendo a granularidade de conjunto, indicador e mês.

- O alvo foi construído temporalmente com `LEAD`, considerando somente meses consecutivos.

- Foram criados atributos atuais, defasagens, médias móveis, histórico de transgressões e variáveis sazonais.

- As partições respeitam a ordem temporal: treino (2021–2023), validação (2024) e teste (2025).

- Imputações e codificações foram ajustadas a partir do treino e aplicadas às demais partições.

- A base final possui `223.184` observações de treino, `75.072` de validação e `69.026` de teste, com `129` atributos alinhados.

- O próximo notebook poderá consumir `X_train`, `y_train`, `X_validation`, `y_validation`, `X_test` e `y_test`.